In [131]:
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

In [132]:
import torch


class customEnv(gym.Env):

    def __init__(self):
        super().__init__()

        self.size = 5
        self.eps = 2
        self.pos = np.array([1, 1], dtype=np.float32)

        self.target_pos = np.array([4, 3], dtype=np.float32)

        self.observation_space = gym.spaces.Dict({
            'observation': gym.spaces.Box(low=-self.size, high=self.size, shape=(2, ), dtype=np.float32)
        })

        self.action_space = gym.spaces.Dict({
            'action': gym.spaces.Box(low=-self.size, high=self.size, shape=(2,), dtype=np.float32),
        })

    def _get_obs(self):
        return {
            'observation': self.pos,
        }

    def _get_info(self):
        return {
            'distance': (np.linalg.norm(self._get_obs()['observation'] - self.target_pos, ord=1))
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.pos = np.array([1, 1], dtype=np.float32)

        self.target_pos = np.array([4, 3], dtype=np.float32)

        obs = self._get_obs()
        info = self._get_info()

        return obs, info

    def step(self, action):
        self.pos = action
        terminated = True if np.array_equal(self.pos, self.target_pos) else False

        truncated = False
        dist = np.linalg.norm(self.pos - self.target_pos, ord=1)
        bonus = 10.0 * np.exp(-(dist**2) / 0.2)
        #reward =  -dist + 10 if dist <= self.eps else -dist**2
        reward = bonus.item() - dist

        obs = self._get_obs()
        info = self._get_info()

        return obs, reward, terminated, truncated, info

In [133]:
gym.register(id='customEnv-v1', entry_point=customEnv)

env = gym.make('customEnv-v1')

/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment customEnv-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [134]:
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor
from torch import nn

import torchrl
from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
    CatTensors
)
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm


In [135]:
torchEnv = torchrl.envs.GymWrapper(env)
torchEnv = TransformedEnv(
    torchEnv,
    Compose(
        StepCounter(),
    ),
)

In [136]:
import torch

torch.manual_seed(0)

import time

from torchrl.envs import GymEnv, StepCounter, TransformedEnv
from tensordict.nn import TensorDictModule as Mod, TensorDictSequential as Seq

In [137]:
torchEnv.action_spec
print(torchEnv.reset())

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)


In [138]:
from torch import nn
from torchrl.modules import MLP, Actor, AdditiveGaussianModule
from tensordict.nn import TensorDictSequential as Seq

# 1. Definiamo la rete per la policy (Actor)
# L'output deve corrispondere alla dimensione delle azioni continue
policy_mlp = MLP(
    out_features=torchEnv.action_spec.shape[-1], 
    num_cells=[64, 64],
    activation_class = nn.Tanh,
    activate_last_layer=False
)

# Trasformiamo la MLP in un Actor che legge "observation" e scrive "action"
policy_module = Actor(
    module=policy_mlp,
    spec=torchEnv.action_spec,
    in_keys=["observation"],
    out_keys=["action"]
)

# 2. Modulo di esplorazione: AdditiveGaussianModule
# Questo è il "gemello" continuo di EGreedyModule
exploration_module = AdditiveGaussianModule(
    spec=torchEnv.action_spec,
    sigma_init=1.0,          # Deviazione standard iniziale (molto rumore)
    sigma_end=0.5,           # Deviazione standard finale (poco rumore)
)

# 3. Componiamo tutto in una sequenza proprio come nel tuo esempio discreto
policy_explore = Seq(policy_module, exploration_module)

# Esempio di test
td = torchEnv.reset()
td = policy_explore(td)
print(f"Azione con rumore (spazio continuo): {td['action']}")

Azione con rumore (spazio continuo): tensor([-0.5871, -0.6909], grad_fn=<AddBackward0>)


In [139]:
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer

init_rand_steps = 100
frames_per_batch = 100
optim_steps = 10
collector = Collector(
    torchEnv,
    policy_explore,
    frames_per_batch=frames_per_batch,
    total_frames=-1,
    init_random_frames=init_rand_steps,
)
rb = ReplayBuffer(storage=LazyTensorStorage(100_000))

from torch.optim import Adam

In [140]:
from torchrl.modules import ValueOperator
from torch.optim import Adam

# Il Critic deve accettare (osservazione + azione)
# Assumiamo obs_dim sia la dimensione delle osservazioni
obs_dim = torchEnv.observation_spec["observation"].shape[-1]
act_dim = torchEnv.action_spec.shape[-1]
print(obs_dim, act_dim)

critic_mlp = MLP(
    in_features=obs_dim + act_dim, 
    out_features=1, 
    num_cells=[128, 256, 128],
    activation_class=nn.Tanh,
    activate_last_layer=False
)

critic_module = ValueOperator(
    module=critic_mlp,
    in_keys=["observation", "action"],
)

2 2


In [141]:
from torchrl.objectives import DDPGLoss, SoftUpdate

# Sostituiamo DQNLoss con DDPGLoss
loss = DDPGLoss(
    actor_network=policy_module,   # La tua policy_module (senza exploration)
    value_network=critic_module,   # Il critic appena creato
    delay_value=True               # Crea automaticamente le reti "target"
)

# Impostiamo il fattore di sconto (gamma)
loss.make_value_estimator(gamma=0.99)

# L'ottimizzatore ora deve gestire i parametri di entrambi (Actor e Critic)
optim = Adam(loss.parameters(), lr=0.02)

# Lo SoftUpdate rimane quasi identico, ma ora aggiorna sia Actor che Critic target
updater = SoftUpdate(loss, eps=0.99)

In [142]:
rollout = torchEnv.rollout(1000, policy_explore)
print(rollout['action'])

tensor([[-0.9801, -0.5208],
        [-0.2656, -0.6724],
        [-1.3492, -0.9885],
        ...,
        [ 0.8651, -0.2896],
        [-0.4198, -2.1465],
        [-1.0501,  1.7699]], grad_fn=<StackBackward0>)


In [ ]:
total_count = 0
total_episodes = 0
t0 = time.time()

for i, data in enumerate(collector):
    # 1. Scrittura dei dati nel replay buffer
    rb.extend(data)
    rewards = data["next", "reward"]
    avg_reward = rewards.mean().item()
    avg_action = data["action"]
    
    print(f"Batch {i}: Reward Medio = {avg_reward:.4f}")
    # Calcoliamo la lunghezza massima raggiunta in questo episodio
    max_length = rb[:]["next", "step_count"].max()
    
    if len(rb) > init_rand_steps:
        for _ in range(optim_steps):
            # 2. Campionamento
            sample = rb.sample(128)
            
            # 3. Calcolo della Loss
            loss_vals = loss(sample)
            
            # NOTA: DDPGLoss restituisce 'loss_actor' e 'loss_value'.
            # Dobbiamo sommarle per fare un unico backward se l'optimizer è condiviso.
            total_loss = loss_vals["loss_actor"] + loss_vals["loss_value"]
            
            total_loss.backward()
            optim.step()
            optim.zero_grad()
            
            # 4. Aggiornamento fattore di esplorazione (Annealing dello sigma)
            # Usiamo i passi reali raccolti
            exploration_module.step(data.numel())
            
            # 5. Update dei parametri target (Soft Update)
            updater.step()
            
            if i % 10 == 0:
                print(f"Passi totali: {total_count}, Loss Actor: {loss_vals['loss_actor']:.4f}, Loss Critic: {loss_vals['loss_value']:.4f}")
        
            total_count += data.numel()
        total_episodes += data["next", "done"].sum()

    # Condizione di uscita basata sulla lunghezza del task
t1 = time.time()

In [ ]:
#see the actions sent to the env
data['action']

tensor([[5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5., 5.],
        [5